In [1]:
import os
import json
import math
import copy
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [24]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)


In [ ]:
DATA_DIR = Path(r"C:\Users\caleb\OneDrive\Desktop\bt4222_grp_project") #change path as needed

ANIME_PATH = DATA_DIR / "anime.csv"
ANIMELIST_PATH = DATA_DIR / "animelist.csv"

# CHANGED: save outputs inside the same project folder
OUTPUT_DIR = DATA_DIR / "data_option_b_animelist"
(OUTPUT_DIR / "mappings").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "interactions").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "meta").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "features" / "item").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "features" / "user").mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------
# Data config
# -----------------------
CHUNK_SIZE = 100_000
MIN_USER_POSITIVES = 5
MIN_ITEM_POSITIVES = 5
MAX_HISTORY_LEN = 30

# status codes
STATUS_WATCHING = 1
STATUS_COMPLETED = 2
STATUS_ON_HOLD = 3
STATUS_DROPPED = 4
STATUS_PLAN_TO_WATCH = 6

# label thresholds
POSITIVE_RATING_THRESHOLD = 8.0
WEAK_POS_SCORE_THRESHOLD = 7.0
LOW_RATING_THRESHOLD = 4.0

WATCHING_HIGH_PROGRESS_THRESHOLD = 0.75
WATCHING_VERY_HIGH_PROGRESS_THRESHOLD = 0.90
DROPPED_EARLY_PROGRESS_THRESHOLD = 0.25

# model
USER_ID_EMB_DIM = 64
HIST_ITEM_EMB_DIM = 64
ITEM_ID_EMB_DIM = 64
USER_HIDDEN_DIM = 128
ITEM_HIDDEN_DIM = 128
OUTPUT_DIM = 64
DROPOUT = 0.2

# train
BATCH_SIZE = 1024
EPOCHS = 10
LR = 1e-3
WEIGHT_DECAY = 1e-6
NUM_WORKERS = 0
PATIENCE = 3

# eval
EVAL_K_LIST = [5, 10, 20]
PRIMARY_K = 10
USE_FULL_CATALOG_EVAL = True
NUM_EVAL_NEGATIVES = 200

# optional scale control for laptop training
MAX_USERS = None         # e.g. set 50000 if needed
MAX_ITEMS = None          # usually leave None

In [26]:
def pad_or_truncate(seq, max_len, pad_value=0):
    seq = seq[:max_len]
    if len(seq) < max_len:
        seq = seq + [pad_value] * (max_len - len(seq))
    return seq


def normalize_episode_count(ep):
    if pd.isna(ep):
        return 0
    if isinstance(ep, str):
        ep = ep.strip()
        if ep == "" or ep.lower() == "unknown":
            return 0
    try:
        ep = int(float(ep))
        return max(ep, 0)
    except:
        return 0


def parse_genres(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [g.strip() for g in str(s).split(",") if g.strip()]


def detect_animelist_rating_col(animelist_path):
    sample = pd.read_csv(animelist_path, nrows=5)
    cols = sample.columns.tolist()
    if "rating" in cols:
        return "rating"
    if "score" in cols:
        return "score"
    raise ValueError(f"Could not find 'rating' or 'score' in animelist.csv. Found: {cols}")


def dcg_at_k(binary_relevance):
    binary_relevance = np.asarray(binary_relevance, dtype=np.float32)
    if binary_relevance.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, binary_relevance.size + 2))
    return float(np.sum(binary_relevance / discounts))


def ndcg_at_k(recommended_items, ground_truth_items, k):
    recommended_k = recommended_items[:k]
    rel = np.array([1.0 if item in ground_truth_items else 0.0 for item in recommended_k], dtype=np.float32)
    dcg = dcg_at_k(rel)

    ideal_count = min(len(ground_truth_items), k)
    if ideal_count == 0:
        return 0.0

    ideal_rel = np.array([1.0] * ideal_count + [0.0] * (k - ideal_count), dtype=np.float32)
    idcg = dcg_at_k(ideal_rel)
    return dcg / idcg if idcg > 0 else 0.0


def compute_progress_ratio(watched_eps, total_eps):
    watched_eps = max(float(watched_eps), 0.0)
    total_eps = float(total_eps)

    if total_eps > 0:
        return min(watched_eps / total_eps, 1.0)

    if watched_eps <= 0:
        return 0.0
    return min(watched_eps / 12.0, 1.0)


def is_positive_interaction(status, score, progress_ratio):
    """
    Strong positive:
        completed + score >= threshold
    Weak positive:
        watching + high progress + decent score
        OR watching + very high progress even if no score
    """
    if status == STATUS_COMPLETED and score >= POSITIVE_RATING_THRESHOLD:
        return True

    if status == STATUS_WATCHING:
        if progress_ratio >= WATCHING_HIGH_PROGRESS_THRESHOLD and score >= WEAK_POS_SCORE_THRESHOLD:
            return True
        if progress_ratio >= WATCHING_VERY_HIGH_PROGRESS_THRESHOLD and score == 0:
            return True

    return False


def is_weak_negative_interaction(status, score, progress_ratio):
    """
    Weak negative:
        dropped early
        or very low score
    """
    if score > 0 and score <= LOW_RATING_THRESHOLD:
        return True

    if status == STATUS_DROPPED and progress_ratio <= DROPPED_EARLY_PROGRESS_THRESHOLD:
        return True

    return False

In [27]:
print("Loading anime.csv...")

anime_cols = [
    "MAL_ID", "Name", "Score", "Genres", "Type", "Episodes",
    "Studios", "Source", "Rating", "Ranked", "Popularity",
    "Members", "Favorites", "Watching", "Completed", "On-Hold",
    "Dropped", "Plan to Watch"
]

anime_df = pd.read_csv(ANIME_PATH, usecols=anime_cols).copy()
anime_df = anime_df.rename(columns={
    "MAL_ID": "anime_id",
    "On-Hold": "On_Hold",
    "Plan to Watch": "Plan_to_Watch"
})

anime_df["anime_id"] = pd.to_numeric(anime_df["anime_id"], errors="coerce")
anime_df = anime_df.dropna(subset=["anime_id"]).copy()
anime_df["anime_id"] = anime_df["anime_id"].astype(int)
anime_df["Episodes"] = anime_df["Episodes"].apply(normalize_episode_count)

anime_df["genre_list"] = anime_df["Genres"].apply(parse_genres)
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(anime_df["genre_list"]).astype(np.float32)

for c in ["Type", "Source", "Rating", "Studios"]:
    anime_df[c] = anime_df[c].fillna("Unknown").astype(str)

type_ohe = pd.get_dummies(anime_df["Type"], prefix="Type", dtype=np.float32)
source_ohe = pd.get_dummies(anime_df["Source"], prefix="Source", dtype=np.float32)
age_rating_ohe = pd.get_dummies(anime_df["Rating"], prefix="AgeRating", dtype=np.float32)

anime_df["Primary_Studio"] = anime_df["Studios"].apply(
    lambda x: x.split(",")[0].strip() if isinstance(x, str) and x.strip() else "Unknown"
)
top_studios = anime_df["Primary_Studio"].value_counts().head(20).index
anime_df["Primary_Studio"] = anime_df["Primary_Studio"].where(
    anime_df["Primary_Studio"].isin(top_studios), "Other"
)
studio_ohe = pd.get_dummies(anime_df["Primary_Studio"], prefix="Studio", dtype=np.float32)

num_cols = [
    "Score", "Episodes", "Ranked", "Popularity",
    "Members", "Favorites", "Watching", "Completed", "On_Hold",
    "Dropped", "Plan_to_Watch"
]

for c in num_cols:
    anime_df[c] = pd.to_numeric(anime_df[c], errors="coerce").fillna(0)

for c in ["Members", "Favorites", "Watching", "Completed", "On_Hold", "Dropped", "Plan_to_Watch"]:
    anime_df[c] = np.log1p(anime_df[c])

item_numeric = anime_df[num_cols].astype(np.float32).values
item_numeric_scaler = StandardScaler()
item_numeric = item_numeric_scaler.fit_transform(item_numeric).astype(np.float32)

item_feature_matrix = np.concatenate([
    item_numeric,
    genre_matrix,
    type_ohe.values.astype(np.float32),
    source_ohe.values.astype(np.float32),
    age_rating_ohe.values.astype(np.float32),
    studio_ohe.values.astype(np.float32),
], axis=1)

anime_episode_map_full = dict(zip(anime_df["anime_id"], anime_df["Episodes"]))

print("anime_df:", anime_df.shape)
print("item_feature_matrix:", item_feature_matrix.shape)

Loading anime.csv...
anime_df: (17562, 20)
item_feature_matrix: (17562, 106)


In [28]:
print("\nScanning animelist.csv and deriving unified labels...")

animelist_rating_col = detect_animelist_rating_col(ANIMELIST_PATH)
animelist_usecols = ["user_id", "anime_id", animelist_rating_col, "watching_status", "watched_episodes"]

valid_anime_in_meta = set(anime_df["anime_id"].tolist())

positive_rows = []
negative_signal_rows = []

user_positive_count = defaultdict(int)
item_positive_count = defaultdict(int)

for chunk_idx, chunk in enumerate(pd.read_csv(ANIMELIST_PATH, usecols=animelist_usecols, chunksize=CHUNK_SIZE), start=1):
    chunk = chunk.dropna(subset=["user_id", "anime_id"]).copy()

    chunk["user_id"] = pd.to_numeric(chunk["user_id"], errors="coerce")
    chunk["anime_id"] = pd.to_numeric(chunk["anime_id"], errors="coerce")
    chunk[animelist_rating_col] = pd.to_numeric(chunk[animelist_rating_col], errors="coerce").fillna(0)
    chunk["watching_status"] = pd.to_numeric(chunk["watching_status"], errors="coerce").fillna(0)
    chunk["watched_episodes"] = pd.to_numeric(chunk["watched_episodes"], errors="coerce").fillna(0)

    chunk = chunk.dropna(subset=["user_id", "anime_id"]).copy()
    chunk["user_id"] = chunk["user_id"].astype(int)
    chunk["anime_id"] = chunk["anime_id"].astype(int)
    chunk[animelist_rating_col] = chunk[animelist_rating_col].astype(float)
    chunk["watching_status"] = chunk["watching_status"].astype(int)
    chunk["watched_episodes"] = chunk["watched_episodes"].astype(float)

    chunk = chunk[chunk["anime_id"].isin(valid_anime_in_meta)].copy()
    if len(chunk) == 0:
        continue

    for row in chunk.itertuples(index=False):
        user_id = row.user_id
        anime_id = row.anime_id
        score = float(getattr(row, animelist_rating_col))
        status = int(row.watching_status)
        watched_eps = float(row.watched_episodes)

        total_eps = anime_episode_map_full.get(anime_id, 0)
        progress_ratio = compute_progress_ratio(watched_eps, total_eps)

        if is_positive_interaction(status, score, progress_ratio):
            positive_rows.append({
                "user_id": user_id,
                "anime_id": anime_id,
                "score": score,
                "watching_status": status,
                "watched_episodes": watched_eps,
                "progress_ratio": progress_ratio
            })
            user_positive_count[user_id] += 1
            item_positive_count[anime_id] += 1

        if is_weak_negative_interaction(status, score, progress_ratio):
            negative_signal_rows.append({
                "user_id": user_id,
                "anime_id": anime_id,
                "score": score,
                "watching_status": status,
                "watched_episodes": watched_eps,
                "progress_ratio": progress_ratio
            })

    if chunk_idx % 20 == 0:
        print(f"Processed {chunk_idx * CHUNK_SIZE:,} animelist rows...")

positive_df = pd.DataFrame(positive_rows)
negative_signal_df = pd.DataFrame(negative_signal_rows)

if positive_df.empty:
    raise ValueError("No positive interactions found. Relax the thresholds.")

valid_users = {u for u, c in user_positive_count.items() if c >= MIN_USER_POSITIVES}
valid_items = {i for i, c in item_positive_count.items() if c >= MIN_ITEM_POSITIVES}

if MAX_USERS is not None:
    valid_users = set(sorted(valid_users)[:MAX_USERS])

if MAX_ITEMS is not None:
    valid_items = set(sorted(valid_items)[:MAX_ITEMS])

positive_df = positive_df[
    positive_df["user_id"].isin(valid_users) &
    positive_df["anime_id"].isin(valid_items)
].copy()

negative_signal_df = negative_signal_df[
    negative_signal_df["user_id"].isin(valid_users) &
    negative_signal_df["anime_id"].isin(valid_items)
].copy()

positive_df = positive_df.drop_duplicates(subset=["user_id", "anime_id"]).copy()
negative_signal_df = negative_signal_df.drop_duplicates(subset=["user_id", "anime_id"]).copy()

print("Unified positives:", positive_df.shape)
print("Negative signal rows:", negative_signal_df.shape)
print("Unique positive users:", positive_df["user_id"].nunique())
print("Unique positive items:", positive_df["anime_id"].nunique())


Scanning animelist.csv and deriving unified labels...
Processed 2,000,000 animelist rows...
Processed 4,000,000 animelist rows...
Processed 6,000,000 animelist rows...
Processed 8,000,000 animelist rows...
Processed 10,000,000 animelist rows...
Processed 12,000,000 animelist rows...
Processed 14,000,000 animelist rows...
Processed 16,000,000 animelist rows...
Processed 18,000,000 animelist rows...
Processed 20,000,000 animelist rows...
Processed 22,000,000 animelist rows...
Processed 24,000,000 animelist rows...
Processed 26,000,000 animelist rows...
Processed 28,000,000 animelist rows...
Processed 30,000,000 animelist rows...
Processed 32,000,000 animelist rows...
Processed 34,000,000 animelist rows...
Processed 36,000,000 animelist rows...
Processed 38,000,000 animelist rows...
Processed 40,000,000 animelist rows...
Processed 42,000,000 animelist rows...
Processed 44,000,000 animelist rows...
Processed 46,000,000 animelist rows...
Processed 48,000,000 animelist rows...
Processed 50,

In [29]:
final_users = sorted(positive_df["user_id"].unique())
final_items = sorted(positive_df["anime_id"].unique())

user2idx = {u: i + 1 for i, u in enumerate(final_users)}
anime2idx = {a: i + 1 for i, a in enumerate(final_items)}

idx2user = {i: u for u, i in user2idx.items()}
idx2anime = {i: a for a, i in anime2idx.items()}

NUM_USERS = len(user2idx) + 1
NUM_ITEMS = len(anime2idx) + 1

with open(OUTPUT_DIR / "mappings" / "user_id_map.json", "w") as f:
    json.dump({str(k): int(v) for k, v in user2idx.items()}, f, indent=2)

with open(OUTPUT_DIR / "mappings" / "anime_id_map.json", "w") as f:
    json.dump({str(k): int(v) for k, v in anime2idx.items()}, f, indent=2)

positive_df["user_idx"] = positive_df["user_id"].map(user2idx)
positive_df["anime_idx"] = positive_df["anime_id"].map(anime2idx)

train_rows = []
val_rows = []
test_rows = []

for user_idx, g in positive_df.groupby("user_idx"):
    # randomize rows within each user
    g = g.sample(frac=1.0, random_state=42)
    rows = g.to_dict("records")
    n = len(rows)

    # Since users have already been filtered to have at least MIN_USER_POSITIVES,
    # and MIN_USER_POSITIVES is currently 5, n should usually be >= 5.
    # But we keep safe handling anyway.

    if n >= 10:
        # Roughly 80/10/10, with at least 1 val and 1 test
        n_val = max(1, int(round(n * 0.10)))
        n_test = max(1, int(round(n * 0.10)))

        # ensure at least 1 train row remains
        while n - n_val - n_test < 1:
            if n_val >= n_test and n_val > 1:
                n_val -= 1
            elif n_test > 1:
                n_test -= 1
            else:
                break

    elif n >= 5:
        # For smaller but still valid users, use a sensible version of 80/10/10
        # that still guarantees 1 val, 1 test, and the rest train
        n_val = 1
        n_test = 1

    elif n == 4:
        n_val = 1
        n_test = 1

    elif n == 3:
        n_val = 1
        n_test = 1

    elif n == 2:
        n_val = 0
        n_test = 1

    else:  # n == 1
        n_val = 0
        n_test = 0

    n_train = n - n_val - n_test

    train_part = rows[:n_train]
    val_part = rows[n_train:n_train + n_val]
    test_part = rows[n_train + n_val:n_train + n_val + n_test]

    train_rows.extend(train_part)
    val_rows.extend(val_part)
    test_rows.extend(test_part)

train_df = pd.DataFrame(train_rows)
val_df = pd.DataFrame(val_rows)
test_df = pd.DataFrame(test_rows)

train_df[["user_idx", "anime_idx", "score"]].to_csv(OUTPUT_DIR / "interactions" / "train.csv", index=False)
val_df[["user_idx", "anime_idx", "score"]].to_csv(OUTPUT_DIR / "interactions" / "val.csv", index=False)
test_df[["user_idx", "anime_idx", "score"]].to_csv(OUTPUT_DIR / "interactions" / "test.csv", index=False)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

train_user_pos = defaultdict(set)
for row in train_df.itertuples(index=False):
    train_user_pos[row.user_idx].add(row.anime_idx)

all_known_positive = defaultdict(set)
for df_ in [train_df, val_df, test_df]:
    for row in df_.itertuples(index=False):
        all_known_positive[row.user_idx].add(row.anime_idx)

heldout_pairs = set()
for df_ in [val_df, test_df]:
    for row in df_.itertuples(index=False):
        heldout_pairs.add((row.user_id, row.anime_id))

Train: (25173563, 8) Val: (3152157, 8) Test: (3152157, 8)
